# RSNA Knee Abnormality Detection: Text-Only Baseline
**When images are unavailable, use reports at both train and test time.**

Pipeline: train.csv reports → XLM-R multilabel classifier → test.csv predictions → submission.csv

## Cell 1: Setup and configuration

In [1]:
import os, sys, re, gc, time, random, warnings, unicodedata
import subprocess
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

def _ensure(pkg, import_name=None):
    try:
        __import__(import_name or pkg)
    except ImportError:
        try:
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], 
                          check=True, timeout=300)
        except Exception as e:
            print(f"[warn] could not install {pkg}: {e}")

_ensure("transformers")
_ensure("torch")
_ensure("scikit-learn", "sklearn")

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup

SEED = 42
MODEL_NAME = "xlm-roberta-base"
MAX_LEN = 384
BATCH_SIZE = 16
EPOCHS = 4
LR = 2e-5
WD = 0.01
VAL_FRAC = 0.12
WARMUP_STEPS = 500

TARGETS = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA", "Lateral OA",
           "PF OA", "Effusion", "Synovitis", "Baker's", "Contusion", "Fracture"]
NT = len(TARGETS)

WORK = "/kaggle/working" if os.path.isdir("/kaggle/working") else "./working"
os.makedirs(WORK, exist_ok=True)

def seed_everything(s):
    random.seed(s)
    np.random.seed(s)
    torch.manual_seed(s)
    torch.cuda.manual_seed_all(s)

seed_everything(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = device.type == "cuda"
print(f"Device: {device} | AMP: {USE_AMP}")

Device: cuda | AMP: True


## Cell 2: Load CSVs and column mapping

In [2]:
COMP_ROOT = "/kaggle/input/competitions/rsna-knee-abnormality-detection"
TRAIN_CSV = os.path.join(COMP_ROOT, "train.csv")
TEST_CSV = os.path.join(COMP_ROOT, "test.csv")
SAMPLE_SUB_CSV = os.path.join(COMP_ROOT, "sample_submission.csv")

def _n(s):
    return re.sub(r"[^a-z0-9]", "", str(s).lower())

ALIASES = {
    "ACL": ["acl"], "MCL": ["mcl"],
    "Medial Meniscus": ["medialmeniscus", "medmeniscus"],
    "Lateral Meniscus": ["lateralmeniscus", "latmeniscus"],
    "Medial OA": ["medialoa"], "Lateral OA": ["lateraloa"],
    "PF OA": ["pfoa", "patellofemoraloa"],
    "Effusion": ["effusion"], "Synovitis": ["synovitis"],
    "Baker's": ["bakers", "baker", "bakerscyst", "bakercyst"],
    "Contusion": ["contusion"], "Fracture": ["fracture"],
}

def map_columns(cols):
    norm_cols = {_n(c): c for c in cols}
    out = {}
    for t, al in ALIASES.items():
        for a in [_n(t)] + al:
            if a in norm_cols:
                out[t] = norm_cols[a]
                break
    return out

# Load training data
train_df = pd.read_csv(TRAIN_CSV)
UID_COL = next((c for c in train_df.columns if _n(c) == "studyinstanceuid"), train_df.columns[0])
REPORT_COL = next((c for c in train_df.columns if c.lower() == "report"),
                  next((c for c in train_df.columns if "report" in c.lower()), None))

if REPORT_COL is None:
    raise ValueError("No report column found in train.csv")

train_df[REPORT_COL] = train_df[REPORT_COL].fillna("").astype(str)
train_df[UID_COL] = train_df[UID_COL].astype(str)
train_df = train_df.drop_duplicates(UID_COL).reset_index(drop=True)

LABEL_COLS = map_columns(train_df.columns)
print(f"train.csv: {train_df.shape}")
print(f"UID column: {UID_COL} | Report column: {REPORT_COL}")
print(f"Mapped label columns: {LABEL_COLS}")

# Load test data
test_df = None
if os.path.isfile(TEST_CSV):
    test_df = pd.read_csv(TEST_CSV)
    test_uid_col = next((c for c in test_df.columns if _n(c) == "studyinstanceuid"), test_df.columns[0])
    test_report_col = next((c for c in test_df.columns if c.lower() == "report"),
                           next((c for c in test_df.columns if "report" in c.lower()), None))
    if test_report_col:
        test_df[test_report_col] = test_df[test_report_col].fillna("").astype(str)
        test_df[test_uid_col] = test_df[test_uid_col].astype(str)
        print(f"test.csv: {test_df.shape} | has reports: YES")
    else:
        print(f"test.csv: {test_df.shape} | has reports: NO (will use priors)")
        test_df = None
else:
    print("test.csv not found")

# Load sample submission to get test UIDs if needed
sample_sub = pd.read_csv(SAMPLE_SUB_CSV) if os.path.isfile(SAMPLE_SUB_CSV) else None
if sample_sub is not None:
    sub_uid_col = next((c for c in sample_sub.columns if _n(c) == "studyinstanceuid"), sample_sub.columns[0])
    TEST_UIDS = sample_sub[sub_uid_col].astype(str).tolist()
    print(f"Test UIDs from sample_submission: {len(TEST_UIDS)}")
else:
    TEST_UIDS = []

train.csv: (4407, 14)
UID column: StudyInstanceUID | Report column: Report
Mapped label columns: {'ACL': 'ACL', 'MCL': 'MCL', 'Medial Meniscus': 'Medial Meniscus', 'Lateral Meniscus': 'Lateral Meniscus', 'Medial OA': 'Medial OA', 'Lateral OA': 'Lateral OA', 'PF OA': 'PF OA', 'Effusion': 'Effusion', 'Synovitis': 'Synovitis', "Baker's": "Baker's", 'Contusion': 'Contusion', 'Fracture': 'Fracture'}
test.csv: (3, 1) | has reports: NO (will use priors)
Test UIDs from sample_submission: 3


## Cell 3: Build soft/hard labels

In [3]:
# Extract gold labels
expl_cols = list(LABEL_COLS.values())
if expl_cols:
    expl_num = train_df[expl_cols].apply(pd.to_numeric, errors="coerce")
    GOLD_MASK = expl_num.notna().any(axis=1).values
else:
    expl_num = pd.DataFrame(index=train_df.index)
    GOLD_MASK = np.zeros(len(train_df), dtype=bool)

print(f"Gold-labeled studies: {int(GOLD_MASK.sum())} / {len(train_df)}")

# Create dummy rule labels (all zeros, since we're not implementing the rules for text-only)
Y_rule = np.zeros((len(train_df), NT), dtype=np.float32)

# Override gold
Y_hard = Y_rule.copy()
for ti, t in enumerate(TARGETS):
    col = LABEL_COLS.get(t)
    if col is None:
        continue
    vals = expl_num[col].values
    m = GOLD_MASK & ~np.isnan(vals)
    Y_hard[m, ti] = (vals[m] > 0.5).astype(np.float32)

PRIORS = np.clip(Y_hard.mean(0), 1e-3, 0.999)
print("Gold positive counts per label:")
print(pd.Series(Y_hard[GOLD_MASK].sum(0).astype(int), index=TARGETS).to_string())
print(f"\nClass priors: {PRIORS.round(3)}")

Gold-labeled studies: 58 / 4407
Gold positive counts per label:
ACL                 24
MCL                  9
Medial Meniscus     26
Lateral Meniscus    23
Medial OA           15
Lateral OA          11
PF OA               21
Effusion            35
Synovitis           27
Baker's             12
Contusion           19
Fracture            18

Class priors: [0.005 0.002 0.006 0.005 0.003 0.002 0.005 0.008 0.006 0.003 0.004 0.004]


## Cell 4: Train/val split

In [4]:
rng = np.random.RandomState(SEED)
n_gold = int(GOLD_MASK.sum())
gold_idx = np.where(GOLD_MASK)[0]
non_gold_idx = np.where(~GOLD_MASK)[0]
rng.shuffle(non_gold_idx)

n_val_rand = max(1, int(VAL_FRAC * len(non_gold_idx)))
val_non_gold = non_gold_idx[:n_val_rand]
tr_non_gold = non_gold_idx[n_val_rand:]

# Put all gold in val if small enough, else split
if n_gold <= 20:
    val_idx = np.sort(np.concatenate([gold_idx, val_non_gold]))
    tr_idx = np.sort(tr_non_gold)
else:
    gold_val = int(0.2 * n_gold)
    gold_tr_idx = gold_idx[:n_gold - gold_val]
    gold_val_idx = gold_idx[n_gold - gold_val:]
    val_idx = np.sort(np.concatenate([gold_val_idx, val_non_gold]))
    tr_idx = np.sort(np.concatenate([gold_tr_idx, tr_non_gold]))

print(f"Train: {len(tr_idx)} | Val: {len(val_idx)} | Gold in val: {int(GOLD_MASK[val_idx].sum())}")

Train: 3875 | Val: 532 | Gold in val: 11


## Cell 5: Dataset and tokenizer

In [5]:
class ReportDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=384):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, i):
        text = str(self.texts[i])
        label = torch.from_numpy(self.labels[i]).float() if self.labels is not None else torch.zeros(NT)
        enc = self.tokenizer(text, truncation=True, max_length=self.max_len, padding="max_length",
                            return_tensors="pt")
        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "labels": label,
        }

print(f"Loading {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=NT)
model = model.to(device)
print(f"Model loaded: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M parameters")

train_texts = train_df[REPORT_COL].values[tr_idx]
train_labels = Y_hard[tr_idx]
val_texts = train_df[REPORT_COL].values[val_idx]
val_labels = Y_hard[val_idx]

train_ds = ReportDataset(train_texts, train_labels, tokenizer, MAX_LEN)
val_ds = ReportDataset(val_texts, val_labels, tokenizer, MAX_LEN)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE * 2, shuffle=False, num_workers=0, pin_memory=True)

print(f"Train loader: {len(train_loader)} batches | Val loader: {len(val_loader)} batches")

Loading xlm-roberta-base...


config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.out_proj.bias    | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model loaded: 278.1M parameters
Train loader: 243 batches | Val loader: 17 batches


## Cell 6: Train the model

In [6]:
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=WARMUP_STEPS,
                                            num_training_steps=total_steps)

pos_weight = torch.tensor(np.clip(np.sqrt((len(tr_idx) - Y_hard[tr_idx].sum(0)) / np.maximum(Y_hard[tr_idx].sum(0), 1)), 1.0, 8.0),
                         dtype=torch.float32, device=device)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight, reduction="mean")

best_score, best_state, history = -1e9, None, []

for epoch in range(1, EPOCHS + 1):
    model.train()
    run_loss, nb = 0.0, 0
    for batch in tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS}", leave=False):
        input_ids = batch["input_ids"].to(device, non_blocking=True)
        attention_mask = batch["attention_mask"].to(device, non_blocking=True)
        labels = batch["labels"].to(device, non_blocking=True)
        
        optimizer.zero_grad(set_to_none=True)
        
        with torch.autocast(device_type=device.type, enabled=USE_AMP):
            logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
            loss = criterion(logits.float(), labels)
        
        if torch.isfinite(loss):
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            run_loss += loss.item()
            nb += 1
    
    # Validate
    model.eval()
    val_preds = []
    with torch.no_grad():
        for batch in tqdm(val_loader, desc="Validating", leave=False):
            input_ids = batch["input_ids"].to(device, non_blocking=True)
            attention_mask = batch["attention_mask"].to(device, non_blocking=True)
            
            with torch.autocast(device_type=device.type, enabled=USE_AMP):
                logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
            
            val_preds.append(torch.sigmoid(logits.float()).cpu().numpy())
    
    P_val = np.concatenate(val_preds, axis=0)
    
    # Compute ROC-AUC (skip single-class)
    aucs = []
    for i in range(NT):
        y = (val_labels[:, i] >= 0.5).astype(int)
        if len(np.unique(y)) >= 2:
            try:
                auc = float(roc_auc_score(y, P_val[:, i]))
                aucs.append(auc)
            except Exception:
                pass
    
    macro_auc = float(np.mean(aucs)) if aucs else np.nan
    val_loss = float(nn.functional.binary_cross_entropy(torch.from_numpy(P_val).clamp(1e-6, 1 - 1e-6),
                                                         torch.from_numpy(val_labels)))
    
    history.append(dict(epoch=epoch, train_loss=run_loss / max(nb, 1), val_loss=val_loss, val_macro_auc=macro_auc))
    
    print(f"Epoch {epoch}: train_loss={run_loss/max(nb,1):.4f} val_loss={val_loss:.4f} macro_auc={macro_auc:.4f}")
    
    if macro_auc > best_score:
        best_score = macro_auc
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        torch.save(best_state, os.path.join(WORK, "best_model.pt"))

if best_state is not None:
    model.load_state_dict(best_state)

pd.DataFrame(history).to_csv(os.path.join(WORK, "history.csv"), index=False)
print("\nTraining complete. Best macro AUC:", round(best_score, 4) if best_score > 0 else "n/a")

Epoch 1/4:   0%|          | 0/243 [00:00<?, ?it/s]

Validating:   0%|          | 0/17 [00:00<?, ?it/s]

Epoch 1: train_loss=0.4114 val_loss=0.0931 macro_auc=0.5172


Epoch 2/4:   0%|          | 0/243 [00:00<?, ?it/s]

Validating:   0%|          | 0/17 [00:00<?, ?it/s]

...

Validating:   0%|          | 0/17 [00:00<?, ?it/s]

Epoch 3: train_loss=0.1462 val_loss=0.0461 macro_auc=0.4837


Epoch 4/4:   0%|          | 0/243 [00:00<?, ?it/s]

Validating:   0%|          | 0/17 [00:00<?, ?it/s]

Epoch 4: train_loss=0.1481 val_loss=0.0468 macro_auc=0.5461

Training complete. Best macro AUC: 0.5461


## Cell 7: Predict on test

In [7]:
model.eval()

# Create test dataset
if test_df is not None and test_report_col in test_df.columns:
    print("Predicting on test reports...")
    test_texts = test_df[test_report_col].values
    test_uids = test_df[test_uid_col].values
    
    # Simple test loader
    test_ds_list = []
    for text in test_texts:
        enc = tokenizer(str(text), truncation=True, max_length=MAX_LEN, padding="max_length", return_tensors="pt")
        test_ds_list.append({
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
        })
    
    P_test = []
    with torch.no_grad():
        for i in tqdm(range(0, len(test_ds_list), BATCH_SIZE), desc="Test prediction"):
            batch_data = test_ds_list[i:i+BATCH_SIZE]
            input_ids = torch.stack([d["input_ids"] for d in batch_data]).to(device)
            attention_mask = torch.stack([d["attention_mask"] for d in batch_data]).to(device)
            
            with torch.autocast(device_type=device.type, enabled=USE_AMP):
                logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
            
            P_test.append(torch.sigmoid(logits.float()).cpu().numpy())
    
    P_test = np.concatenate(P_test, axis=0)
    pred = P_test
    pred_uids = test_uids

else:
    print("No test reports available. Using class priors for all test studies.")
    pred = np.tile(PRIORS, (len(TEST_UIDS), 1)).astype(np.float32)
    pred_uids = TEST_UIDS

pred = np.clip(pred, 1e-4, 1 - 1e-4)

No test reports available. Using class priors for all test studies.


## Cell 8: Write submission

In [8]:
sub = pd.DataFrame(pred, columns=TARGETS)
sub.insert(0, "StudyInstanceUID", pred_uids)

# Match sample submission column order if possible
if sample_sub is not None:
    col_map = map_columns([c for c in sample_sub.columns if _n(c) != "studyinstanceuid"])
    if len(col_map) == NT:
        sub = sub.rename(columns={t: col_map[t] for t in TARGETS})
        col_order = [sub_uid_col] + [c for c in sample_sub.columns if c != sub_uid_col]
        sub = sub[[c for c in col_order if c in sub.columns]]

sub.to_csv(os.path.join(WORK, "submission.csv"), index=False)
print(f"\nSubmission saved: {len(sub)} rows, {len(sub.columns)} columns")
print(f"Columns: {list(sub.columns)}")
print(f"Pred range: [{pred.min():.4f}, {pred.max():.4f}]")
print(f"Mean pred per label:\n{sub.iloc[:, 1:].mean().round(3).to_string()}")
print("\n" + sub.head().to_string())


Submission saved: 3 rows, 13 columns
Columns: ['StudyInstanceUID', 'ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus', 'Medial OA', 'Lateral OA', 'PF OA', 'Effusion', 'Synovitis', "Baker's", 'Contusion', 'Fracture']
Pred range: [0.0020, 0.0079]
Mean pred per label:
ACL                 0.005
MCL                 0.002
Medial Meniscus     0.006
Lateral Meniscus    0.005
Medial OA           0.003
Lateral OA          0.002
PF OA               0.005
Effusion            0.008
Synovitis           0.006
Baker's             0.003
Contusion           0.004
Fracture            0.004

                                                   StudyInstanceUID       ACL       MCL  Medial Meniscus  Lateral Meniscus  Medial OA  Lateral OA     PF OA  Effusion  Synovitis   Baker's  Contusion  Fracture
0  1.2.826.0.1.3680043.8.498.10047035057544427318018579121635276191  0.005446  0.002042           0.0059          0.005219   0.003404    0.002496  0.004765  0.007942   0.006127  0.002723   0.004311  0.004084
1  

## Cell 9: Summary

In [9]:
print("\n" + "="*60)
print("TEXT-ONLY BASELINE SUMMARY")
print("="*60)
print(f"Train studies: {len(tr_idx)} | Val studies: {len(val_idx)}")
print(f"Model: {MODEL_NAME} | Backbone params: {sum(p.numel() for p in model.parameters())/1e6:.1f}M")
print(f"Val macro ROC-AUC: {round(best_score, 4) if best_score > 0 else 'n/a'}")
print(f"Test predictions: {len(pred)} studies")
print(f"Submission file: {os.path.join(WORK, 'submission.csv')}")
print("="*60)


TEXT-ONLY BASELINE SUMMARY
Train studies: 3875 | Val studies: 532
Model: xlm-roberta-base | Backbone params: 278.1M
Val macro ROC-AUC: 0.5461
Test predictions: 3 studies
Submission file: /kaggle/working/submission.csv
